# 20 · A spinning fan — rotating-frame H(div)-HDG 🌀💨

We simulate the **flow around a three-blade fan** in **3D**. Sitting in the **co-rotating
frame** (spinning with the blades at angular velocity $\Omega$) turns a moving-boundary
problem into a **steady** one: the blades are fixed walls, the far field appears to swirl,
and two extra forces appear — **Coriolis** and **centrifugal**. We discretise with an
**exactly divergence-free H(div)-HDG** velocity (unit 13) and keep the **Reynolds number
deliberately tiny** so a coarse mesh suffices.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "anywidget"], check=True)

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.krylovspace import GMRes
from ngsolve.webgui import Draw
import numpy as np
import matplotlib.pyplot as plt

## 1. The fan — three blades in a cylinder

The geometry is **constructive** (unit 2): a **hub** (small cylinder) carries **three pitched
blades** at $120^\circ$, each a thin box tilted by a *pitch* angle so that spinning pushes
air along the axis. We subtract that fan from an enclosing **cylinder** of air. The blades are
thin, so we **refine the mesh on their faces** — everything else stays coarse.

In [ ]:
R_dom, H_dom = 4.0, 2.2                                  # enclosure: radius, half-height (axis = z)
r_hub, L_blade, pitch, half_t = 0.5, 2.6, 30, 0.20      # hub radius, blade length, pitch angle, half-thickness

cyl = Cylinder(Pnt(0, 0, -H_dom), Z, r=R_dom, h=2*H_dom)
hub = Cylinder(Pnt(0, 0, -0.6), Z, r=r_hub, h=1.2)
def blade(angle_deg):
    b = Box(Pnt(r_hub-0.12, -0.55, -half_t), Pnt(L_blade, 0.55, half_t))   # radial x, chord y, thin z
    b = b.Rotate(Axis(Pnt(0, 0, 0), X), pitch)                             # pitch about the radial axis
    return b.Rotate(Axis(Pnt(0, 0, 0), Z), angle_deg)                      # place around the axis

fan = hub
for i in range(3):
    fan = fan + blade(i*120)
fan.faces.name = "blades"                                # no-slip walls (rotate with the frame)
cyl.faces.name = "outer"                                 # the enclosure
air = cyl - fan; air.name = "air"
for f in air.faces:
    if f.name == "blades":
        f.maxh = 0.25                                    # resolve the thin blades + their boundary layer

mesh = Mesh(OCCGeometry(air).GenerateMesh(maxh=1.1)); mesh.Curve(2)
print(f"mesh: {mesh.ne} tetrahedra,  boundaries {sorted(set(mesh.GetBoundaries()))}")
Draw(mesh, clipping={"function": True, "pnt": (0, 0, 0), "vec": (0, 0, 1)})

## 2. The rotating-frame model

In a frame spinning with angular velocity $\boldsymbol\Omega=\omega\,\mathbf e_z$ the steady
incompressible flow obeys, for velocity $\mathbf u$ and pressure $p$,
$$ -\nu\,\Delta\mathbf u \;+\; \underbrace{2\,\boldsymbol\Omega\times\mathbf u}_{\text{Coriolis}}
   \;+\; \nabla p \;=\; -\,\underbrace{\boldsymbol\Omega\times(\boldsymbol\Omega\times\mathbf r)}_{\text{centrifugal}},
   \qquad \nabla\!\cdot\!\mathbf u = 0 . $$
The **centrifugal** term is a gradient, $\boldsymbol\Omega\times(\boldsymbol\Omega\times\mathbf r)=
-\tfrac12\nabla(\omega^2 r^2)$, so it is **absorbed into the pressure** and never appears
explicitly. **Boundary conditions:** the **blades** are fixed in this frame → **no-slip**
$\mathbf u=0$; the room air (at rest in the lab) appears to **swirl**, so on the **outer**
wall $\mathbf u = -\boldsymbol\Omega\times\mathbf r = (\omega y,\,-\omega x,\,0)$. We keep
the flow **slow** (large $\nu$): a tiny Reynolds number lets convection be dropped (Stokes).

In [ ]:
order, nu, omega = 1, 2.0, 1.0                           # low order + large nu  ->  tiny Reynolds number
walls = "outer|blades"
V    = HDiv(mesh, order=order, dirichlet=walls, dgjumps=True)     # exactly divergence-free velocity
Vhat = TangentialFacetFESpace(mesh, order=order, dirichlet=walls) # HDG tangential trace
Q    = L2(mesh, order=order-1)                                    # pressure
X = V * Vhat * Q
(u, uhat, p), (v, vhat, q) = X.TnT()
print(f"H(div)-HDG system: {X.ndof} dofs")

## 3. Assemble & solve — explicit Coriolis, symmetric factorisation

The viscous + pressure part is the **symmetric** H(div)-HDG Stokes operator (unit 13); we
factor it **once** with `sparsecholesky`. The **Coriolis** term $2\,\Omega\times
\mathbf u$ is *skew*, so adding it would break symmetry — instead we keep it **out of the
factored matrix** and let **GMRes** apply it, using the symmetric Stokes solve as a
preconditioner. A few dozen iterations suffice.

In [ ]:
n = specialcf.normal(3); h = specialcf.mesh_size
def tang(w): return w - (w*n)*n
def cross(a, b): return CF((a[1]*b[2]-a[2]*b[1], a[2]*b[0]-a[0]*b[2], a[0]*b[1]-a[1]*b[0]))
dS = dx(element_boundary=True); alpha = 4; eps = 1e-8; Om = CF((0, 0, omega))

def stokes(bf):                                          # symmetric H(div)-HDG Stokes
    bf += nu*InnerProduct(Grad(u), Grad(v))*dx
    bf += nu*(-InnerProduct(Grad(u)*n, tang(v-vhat)) - InnerProduct(Grad(v)*n, tang(u-uhat))
              + alpha*order*order/h*InnerProduct(tang(u-uhat), tang(v-vhat)))*dS
    bf += (-div(u)*q - div(v)*p - eps*p*q)*dx            # incompressibility (+ pressure regularisation)

a0 = BilinearForm(X, symmetric=True); stokes(a0); a0.Assemble()        # factor this (symmetric)
A  = BilinearForm(X, symmetric=False); stokes(A)                       # full operator (with Coriolis)
A += 2*InnerProduct(cross(Om, u), v)*dx
A.Assemble()

gfu = GridFunction(X)
gfu.components[1].Set(CF((omega*y, -omega*x, 0)), definedon=mesh.Boundaries("outer"))   # swirl BC
pre = a0.mat.Inverse(X.FreeDofs(), inverse="sparsecholesky")          # symmetric Stokes preconditioner
res = (-A.mat*gfu.vec).Evaluate()                                     # residual of the Dirichlet lift
gfu.vec.data += GMRes(A=A.mat, b=res, pre=pre, maxsteps=400, tol=1e-8, printrates=False)
vel, pres = gfu.components[0], gfu.components[2]
print(f"rms |u| = {sqrt(Integrate(InnerProduct(vel,vel)*dx, mesh)/Integrate(CF(1)*dx, mesh)):.2f}"
      f"   ||div u|| = {sqrt(Integrate(div(vel)*div(vel)*dx, mesh)):.1e}  (exactly divergence-free)")

## 4. The flow — speed inside the fan

We draw the **speed** $|\mathbf u|$, clipped through the axis so we can see *inside* the
enclosure: the swirl grows with radius, and the blades carve their no-slip wakes into it.

In [ ]:
speed = sqrt(InnerProduct(vel, vel))
Draw(speed, mesh, "speed", clipping={"function": True, "pnt": (0, 0, 0), "vec": (0, 1, 0)})

**Streamlines in the rotation plane.** A 2-D slice at $z=0$ shows the in-plane velocity as
streamlines, swirling around the three blades. (We project the H(div) field onto a smooth
`VectorH1` first, purely so the slice samples cleanly.)

In [ ]:
vsmooth = GridFunction(VectorH1(mesh, order=2)); vsmooth.Set(vel)
N = 150; g = np.linspace(-R_dom, R_dom, N)
U = np.full((N, N), np.nan); Vv = U.copy(); M = U.copy()
for i, yy in enumerate(g):
    for j, xx in enumerate(g):
        if xx*xx + yy*yy <= (R_dom*0.99)**2:
            try:
                w = vsmooth(mesh(float(xx), float(yy), 0.0))
                U[i, j], Vv[i, j], M[i, j] = w[0], w[1], np.hypot(w[0], w[1])
            except Exception:
                pass
fig, ax = plt.subplots(figsize=(6, 6))
ax.contourf(g, g, np.ma.masked_invalid(M), levels=22, cmap="viridis")
ax.streamplot(g, g, np.nan_to_num(U), np.nan_to_num(Vv), color="white", density=1.8, linewidth=0.6, arrowsize=0.7)
th = np.linspace(0, 2*np.pi, 200); ax.plot(R_dom*np.cos(th), R_dom*np.sin(th), "k", lw=1)
ax.set_aspect("equal"); ax.axis("off"); ax.set_title("rotating frame — swirl plane z = 0")
plt.show()

## 5. Back to the lab — what the fan actually does

*(Supplementary.)* Add the frame velocity back, $\mathbf u_{\text{lab}}=\mathbf u+\boldsymbol
\Omega\times\mathbf r$: now the **far field is at rest** (the room) and the **blades drag the
air**, exactly the picture of a fan pushing a breeze. The same field, two viewpoints.

In [ ]:
u_lab = vel + CF((-omega*y, omega*x, 0))
Draw(sqrt(InnerProduct(u_lab, u_lab)), mesh, "lab speed",
     clipping={"function": True, "pnt": (0, 0, 0), "vec": (0, 1, 0)})

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("19-thermoelasticity", "19 · A warm chocolate bar bends 🍫🔥")
    _next = None
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))